In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Data

In [116]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
df_original = pd.read_csv('data/Rainfall.csv')

In [117]:
df_train.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


- we remove 'id' column
- 'rainfall' variable is our target

In [118]:
train = df_train.copy()
train.drop(['id'], axis=1, inplace=True)

In [119]:
train.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [120]:
X_train = train.drop('rainfall', axis=1)
Y_train = train['rainfall']
target = 'rainfall'

In [121]:
df_original.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1,1025.9,19.9,18.3,16.8,13.1,72,49,yes,9.3,80.0,26.3
1,2,1022.0,21.7,18.9,17.2,15.6,81,83,yes,0.6,50.0,15.3
2,3,1019.7,20.3,19.3,18.0,18.4,95,91,yes,0.0,40.0,14.2
3,4,1018.9,22.3,20.6,19.1,18.8,90,88,yes,1.0,50.0,16.9
4,5,1015.9,21.3,20.7,20.2,19.9,95,81,yes,0.0,40.0,13.7


In [122]:
df_original.columns = df_original.columns.str.strip()
original = df_original.copy()
original['rainfall'] = original['rainfall'].map({'no': 0, 'yes': 1})

In [123]:
X_original = original.drop('rainfall', axis=1)
Y_original = original['rainfall']

## Adding weather information from previous days

In [136]:
train_shifted1 = train.shift(1)
train_shifted1 = train_shifted1.add_prefix("prev1_")
train_shifted1.head()

,prev1_day,prev1_pressure,prev1_maxtemp,prev1_temparature,prev1_mintemp,prev1_dewpoint,prev1_humidity,prev1_cloud,prev1_sunshine,prev1_winddirection,prev1_windspeed,prev1_rainfall
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1.0
2,2.0,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1.0
3,3.0,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1.0
4,4.0,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1.0


In [125]:
train_prev_1 = pd.concat([train, train_shifted1], axis=1)
train_prev_1.drop(['prev1_day'], axis=1, inplace=True)
train_prev_1 = train_prev_1.dropna().reset_index(drop=True) # removing the first record with no information from the previous day

In [126]:
X_train_prev_1 = train_prev_1.drop('rainfall', axis=1)
Y_train_prev_1 = train_prev_1['rainfall']

- 'train_prev_1' - the training dataset with additional columns containing weather parameters from the previous day

In [127]:
train_prev_1.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,prev1_maxtemp,prev1_temparature,prev1_mintemp,prev1_dewpoint,prev1_humidity,prev1_cloud,prev1_sunshine,prev1_winddirection,prev1_windspeed,prev1_rainfall
0,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,...,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1.0
1,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,...,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1.0
2,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,...,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1.0
3,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,...,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1.0
4,6,1022.7,20.6,18.6,16.5,12.5,79.0,81.0,0.0,20.0,...,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0.0


In [135]:
train_shifted2 = train.shift(2)
train_shifted2 = train_shifted2.add_prefix("prev2_")
train_shifted2.head()

,prev2_day,prev2_pressure,prev2_maxtemp,prev2_temparature,prev2_mintemp,prev2_dewpoint,prev2_humidity,prev2_cloud,prev2_sunshine,prev2_winddirection,prev2_windspeed,prev2_rainfall
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1.0
3,2.0,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1.0
4,3.0,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1.0


In [129]:
train_prev_2 = pd.concat([train_prev_1, train_shifted2.drop(index=0).reset_index(drop=True)], axis=1)
train_prev_2.drop(['prev2_day'], axis=1, inplace=True)
train_prev_2 = train_prev_2.dropna().reset_index(drop=True) # removing the second record with no information from the previous days

In [130]:
X_train_prev_2 = train_prev_2.drop('rainfall', axis=1)
Y_train_prev_2 = train_prev_2['rainfall']

- 'train_prev_2' – the training dataset with additional columns containing weather parameters from the previous two days

In [131]:
train_prev_2.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,prev2_maxtemp,prev2_temparature,prev2_mintemp,prev2_dewpoint,prev2_humidity,prev2_cloud,prev2_sunshine,prev2_winddirection,prev2_windspeed,prev2_rainfall
0,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,...,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1.0
1,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,...,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1.0
2,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,...,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1.0
3,6,1022.7,20.6,18.6,16.5,12.5,79.0,81.0,0.0,20.0,...,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1.0
4,7,1022.8,19.5,18.4,15.3,11.3,56.0,46.0,7.6,20.0,...,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0.0


- we add columns with average parameters from the two previous days omitting 'winddirection' and 'rainfall'

In [137]:
train_shifted1.drop(['prev1_day', 'prev1_winddirection', 'prev1_rainfall'], axis=1, inplace=True)
train_shifted2.drop(['prev2_day', 'prev2_winddirection', 'prev2_rainfall'], axis=1, inplace=True)

train_shifted1.columns = train_shifted1.columns.str.removeprefix('prev1_')
train_shifted2.columns = train_shifted2.columns.str.removeprefix('prev2_')

In [138]:
train_avg = pd.concat([train_shifted1, train_shifted2]).groupby(level=0).mean()
train_avg[(train_shifted1.isna()) | (train_shifted2.isna())] = np.nan
train_avg = train_avg.add_prefix("prev_avg_")
train_avg = train_avg.dropna().reset_index(drop=True)
train_avg

,prev_avg_pressure,prev_avg_maxtemp,prev_avg_temparature,prev_avg_mintemp,prev_avg_dewpoint,prev_avg_humidity,prev_avg_cloud,prev_avg_sunshine,prev_avg_windspeed
0,1018.45,18.70,18.75,17.85,17.40,91.0,89.5,0.55,19.55
1,1021.80,17.80,16.50,15.20,12.35,85.0,69.0,4.15,20.00
2,1018.75,18.75,16.95,15.75,13.05,85.0,71.0,4.15,26.85
3,1017.60,19.70,18.10,16.05,13.20,73.5,70.0,1.80,30.20
4,1022.25,20.95,18.50,15.85,11.05,65.5,63.0,1.80,20.25
...,...,...,...,...,...,...,...,...,...
2183,1016.35,19.65,17.60,16.45,15.65,78.5,86.0,1.20,34.30
2184,1015.20,22.05,19.70,18.35,18.80,87.5,88.0,0.05,26.35
2185,1013.50,20.20,18.95,17.70,17.60,94.0,88.0,0.05,28.70
2186,1012.85,18.10,16.80,15.30,13.95,85.0,83.5,2.50,34.10


In [139]:
train_prev_avg = pd.concat([train_prev_2, train_avg], axis=1)
train_prev_avg.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,prev2_rainfall,prev_avg_pressure,prev_avg_maxtemp,prev_avg_temparature,prev_avg_mintemp,prev_avg_dewpoint,prev_avg_humidity,prev_avg_cloud,prev_avg_sunshine,prev_avg_windspeed
0,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,...,1.0,1018.45,18.70,18.75,17.85,17.40,91.0,89.5,0.55,19.55
1,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,...,1.0,1021.80,17.80,16.50,15.20,12.35,85.0,69.0,4.15,20.00
2,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,...,1.0,1018.75,18.75,16.95,15.75,13.05,85.0,71.0,4.15,26.85
3,6,1022.7,20.6,18.6,16.5,12.5,79.0,81.0,0.0,20.0,...,1.0,1017.60,19.70,18.10,16.05,13.20,73.5,70.0,1.80,30.20
4,7,1022.8,19.5,18.4,15.3,11.3,56.0,46.0,7.6,20.0,...,0.0,1022.25,20.95,18.50,15.85,11.05,65.5,63.0,1.80,20.25


- 'train_prev_avg' - the training dataset with additional columns containing weather parameters from the previous two days and their average values

In [140]:
X_train_prev_avg = train_prev_avg.drop('rainfall', axis=1)
Y_train_prev_avg = train_prev_avg['rainfall']

## Checking importance of new columns

### Random Forest

In [88]:
from sklearn.ensemble import RandomForestClassifier

**Original data**

In [141]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, Y_train)

importances = clf.feature_importances_
features_importance_org = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)

print(features_importance_org)

cloud            0.303654
sunshine         0.158596
humidity         0.096387
dewpoint         0.061703
windspeed        0.060770
day              0.058626
maxtemp          0.058191
pressure         0.055699
mintemp          0.054301
temparature      0.054286
winddirection    0.037788
dtype: float64


**Data with columns from the previous two days and their average values**

In [142]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_prev_avg, Y_train_prev_avg)

importances = clf.feature_importances_
features_importance_prev = pd.Series(importances, index=X_train_prev_avg.columns).sort_values(ascending=False)

print(features_importance_prev)

cloud                   0.198138
sunshine                0.147906
humidity                0.058710
dewpoint                0.026705
windspeed               0.023581
pressure                0.021445
temparature             0.020895
maxtemp                 0.019444
day                     0.018754
prev_avg_sunshine       0.017815
prev_avg_cloud          0.017392
prev2_windspeed         0.017305
prev_avg_windspeed      0.017161
prev1_sunshine          0.016854
prev1_windspeed         0.016780
prev1_cloud             0.016702
mintemp                 0.016556
prev2_pressure          0.016542
prev_avg_pressure       0.016354
prev2_maxtemp           0.016282
prev_avg_maxtemp        0.016277
prev_avg_dewpoint       0.016161
prev_avg_humidity       0.015952
prev1_temparature       0.015176
prev2_mintemp           0.015115
prev_avg_mintemp        0.014903
prev_avg_temparature    0.014688
prev1_dewpoint          0.014548
prev2_temparature       0.014456
prev1_mintemp           0.014346
prev2_dewp

**Data with columns from the previous two days**

In [143]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_prev_2, Y_train_prev_2)

importances = clf.feature_importances_
features_importance_prev_2 = pd.Series(importances, index=X_train_prev_2.columns).sort_values(ascending=False)

print(features_importance_prev_2)

cloud                  0.229534
sunshine               0.126720
humidity               0.082463
dewpoint               0.027494
maxtemp                0.026859
windspeed              0.026026
pressure               0.025765
temparature            0.023095
prev1_windspeed        0.022994
day                    0.022885
prev2_windspeed        0.021726
prev1_sunshine         0.021570
prev2_maxtemp          0.020541
prev1_dewpoint         0.020529
prev2_pressure         0.020511
prev1_cloud            0.019809
mintemp                0.019440
prev2_temparature      0.019403
prev2_dewpoint         0.018927
prev1_maxtemp          0.018743
prev1_temparature      0.018684
prev1_mintemp          0.018471
prev2_mintemp          0.018300
prev1_pressure         0.018059
prev1_humidity         0.017263
prev2_sunshine         0.016294
winddirection          0.015415
prev2_humidity         0.014709
prev2_cloud            0.014655
prev1_winddirection    0.014135
prev2_winddirection    0.012453
prev1_ra

**Data with columns from the previous day**

In [144]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_prev_1, Y_train_prev_1)

importances = clf.feature_importances_
features_importance_prev_1 = pd.Series(importances, index=X_train_prev_1.columns).sort_values(ascending=False)

print(features_importance_prev_1)

cloud                  0.233276
sunshine               0.161569
humidity               0.072822
dewpoint               0.037951
maxtemp                0.037069
windspeed              0.034967
pressure               0.033087
day                    0.032508
temparature            0.031632
prev1_windspeed        0.030806
mintemp                0.030251
prev1_sunshine         0.030110
prev1_maxtemp          0.028938
prev1_dewpoint         0.028397
prev1_pressure         0.027957
prev1_mintemp          0.027715
prev1_cloud            0.025636
prev1_temparature      0.024830
prev1_humidity         0.023452
winddirection          0.021906
prev1_winddirection    0.019879
prev1_rainfall         0.005242
dtype: float64
